# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Paper Finding 1

Finding:

The paper reports that machine learning improves prioritization compared with a simple baseline.

Methodology Question:

How was the label defined?

Was the model predicting a future observed outcome or a proxy label derived from current data?

Understanding the label helps interpret how useful the model may be in practice.

---

# Paper Finding 2

Finding:

The paper reports strong model performance.

Methodology Question:

Does the validation design support the claim?

Were examples from the same client separated between training and testing?

This helps determine whether the reported performance is likely to generalize.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

# Original Validation

Week 5 used a standard train-test split.

This provides a useful starting point but may allow related examples to appear in both training and testing sets.

In [5]:
#Honest Split
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

X = df[
    [
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "content_age_days"
    ]
].fillna(0)

y = df["is_declining_label"]

from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [6]:
# training again

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(
    random_state=42
)

rf.fit(X_train, y_train)

group_probs = rf.predict_proba(
    X_test
)[:,1]

group_auc = roc_auc_score(
    y_test,
    group_probs
)

print("Grouped ROC-AUC:", group_auc)

Grouped ROC-AUC: 0.6046287537590536


In [8]:
# From week 5 - w05
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_test)[:,1]

model_auc = roc_auc_score(
    y_test,
    rf_probs
)

print("Standard Split ROC AUC:", model_auc)

Standard Split ROC AUC: 0.7270703192839789


In [9]:
#Comparison Table

comparison = pd.DataFrame(
    {
        "Validation": [
            "Standard Split",
            "Grouped Split"
        ],
        "ROC_AUC": [
            model_auc,
            group_auc
        ]
    }
)

comparison

,Validation,ROC_AUC
0,Standard Split,0.727070
1,Grouped Split,0.604629


## Before and After
The grouped validation result is a more conservative estimate because entire client groups are held out.
If performance decreases, that does not mean the model failed.
It means the evaluation became more realistic.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

# Leakage Audit

Included Features

- impressions_90d
- clicks_90d
- sessions_90d
- ctr
- avg_position
- content_age_days

Excluded Features

- trend_direction
- is_declining_label
- future-window measurements
- target-derived fields
- product decision outputs

All included features would be available before the review decision is made.

In [13]:
# dISPLAY FEATURES USED
list(X.columns)

['impressions_90d',
 'clicks_90d',
 'sessions_90d',
 'ctr',
 'avg_position',
 'content_age_days']

In [14]:
#Building Failure Examples
print(len(y_test))
print(len(group_probs))

6000
6163


In [24]:
#Rerunning the grouped Split Section
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

In [25]:
#Trainign Grouped Model
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42
)

rf.fit(
    X_train_group,
    y_train_group
)

group_probs = rf.predict_proba(
    X_test_group
)[:,1]

In [26]:
#Verifying Lengths
print(len(y_test_group))
print(len(group_probs))

6163
6163


In [27]:
#Creating prediction Dataframes
pred_df = pd.DataFrame({
    "actual": y_test_group.reset_index(drop=True),
    "predicted_prob": group_probs
})

In [23]:
# Generating errors
pred_df["prediction"] = (
    pred_df["predicted_prob"] > 0.5
).astype(int)

errors = pred_df[
    pred_df["actual"] != pred_df["prediction"]
]

errors.head(10)

,actual,predicted_prob,prediction
4,1,0.44,0
7,1,0.44,0
8,1,0.36,0
9,0,0.85,1
12,1,0.27,0
17,1,0.14,0
18,1,0.43,0
19,0,0.67,1
21,1,0.43,0
22,0,0.60,1


# Failure Examples

False Positives

The model occasionally predicts decline risk for pages that are not labeled as declining.

These pages share some characteristics with declining pages but do not ultimately receive the label.

False Negatives

The model occasionally misses pages that belong to the decline-risk label.

These examples may contain information not fully represented by the current feature set.

These errors demonstrate that the model should support human review rather than replace it.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# Claim Rewrite

Original Claim

The model identifies the pages that should be refreshed.

Rewritten Claim

The model provides a ranked list of pages that may deserve review based on observed search and engagement signals.

---

Original Claim

Refreshing these pages will improve traffic.

Rewritten Claim

These pages show characteristics associated with review opportunities, but the analysis does not establish that refreshing content will cause traffic improvements.

---

Original Claim

The model finds the causes of decline.

Rewritten Claim

The model identifies patterns associated with the decline proxy used in this analysis.

## Self-check

Before you submit, confirm each line honestly:

- [y ] Every section above is filled — markdown thinking AND the code that backs it
- [ y] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ y] No client names, URLs, or private queries anywhere
- [ y] My claims use careful words: observed, measured, directional, decision-support
- [ y] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.